# E-Commerce Customer Retention & Churn Analysis

**Colab-ready notebook** executing the full 500K+ transaction pipeline.

This notebook loads the UCI Online Retail dataset, cleans it, computes business
metrics (Churn Rate, AOV, LTV), and renders inline Seaborn visualizations.

---

## Setup

If running on **Google Colab**, clone the repo first:
```python
!git clone https://github.com/<YOUR_USERNAME>/Ecommerce-Analysis.git
%cd Ecommerce-Analysis
!pip install -r requirements.txt -q
```

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure src is importable (works in Colab or local)
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', font_scale=1.1)
print('✓ Imports ready')

## 1. Load Data

In [ ]:
from src.data_loader import load_data

# Set full=True to load the full 541K-row dataset (requires the file in data/)
raw = load_data(full=True)
raw.head()

In [ ]:
print(f'Shape: {raw.shape}')
print(f'\nDtypes:\n{raw.dtypes}')
print(f'\nNull counts:\n{raw.isnull().sum()}')
print(f'\nDuplicate rows: {raw.duplicated().sum()}')

## 2. Data Cleaning

In [ ]:
from src.cleaner import clean_data

df = clean_data(raw)
df.head()

In [ ]:
print(f'Cleaned shape: {df.shape}')
print(f'Null remaining: {df.isnull().sum().sum()}')
print(f'Date range: {df["InvoiceDate"].min()} — {df["InvoiceDate"].max()}')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')

## 3. Business Metrics

In [ ]:
from src.metrics import compute_churn_rate, compute_aov, compute_ltv

churn = compute_churn_rate(df, window_days=60)
aov = compute_aov(df)
ltv = compute_ltv(df)

print(f"Churn Rate: {churn['churn_rate']:.2f}%")
print(f"Overall AOV: ${aov['overall_aov']:,.2f}")
print(f"Mean LTV: ${ltv['overall_ltv']:,.2f}")
print(f"Median LTV: ${ltv['median_ltv']:,.2f}")

## 4. Visualizations

### 4.1 Category Churn Risk & Drop-off

In [ ]:
from src.visualization import plot_category_churn, plot_ltv_distribution, plot_monthly_aov

plot_category_churn(churn, save_dir='assets/images')

from IPython.display import Image
Image('assets/images/category_churn_risk.png')

### 4.2 Customer LTV & Spend Distribution

In [ ]:
plot_ltv_distribution(ltv, save_dir='assets/images')
Image('assets/images/ltv_distribution.png')

### 4.3 Monthly AOV Trend

In [ ]:
plot_monthly_aov(aov, save_dir='assets/images')
Image('assets/images/monthly_aov_trend.png')

## 5. LTV Segment Deep-Dive

In [ ]:
print('LTV Segment Summary:')
ltv['segment_summary']

In [ ]:
print('Top 10 Most Valuable Customers:')
ltv['customer_stats'].nlargest(10, 'LTV')[['total_spent', 'order_count', 'avg_order_value', 'lifespan_years', 'LTV']]

---

*Analysis complete. All charts saved to `assets/images/`.*